In [6]:
import pandas as pd
from datetime import datetime
import re

# Load dataset
df = pd.read_csv("gold-dataset-sinha-khandait.csv")

# Regex pattern to find YYYY-MM-DD in URLs
date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")

def fix_date(row):
    original_date = str(row["Dates"])
    url = str(row["URL"])

    # Regex pattern for YYYY-MM-DD
    valid_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    url_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")

    # Try to extract a valid date from the URL
    match = url_pattern.search(url)
    extracted_date = match.group(0) if match else None

    def normalize(date_str):
        try:
            return pd.to_datetime(date_str, errors="coerce").strftime("%Y-%m-%d")
        except Exception:
            return None

    normalized = normalize(original_date)

    # Check if normalized year is reasonable
    if normalized:
        year = int(normalized[:4])
        if 1900 <= year <= datetime.now().year:
            # Valid year, keep it
            return pd.Series([normalized, False])
        else:
            # Year out of range, replace with URL date if available
            if extracted_date:
                return pd.Series([extracted_date, True])
            else:
                return pd.Series([normalized, False])
    else:
        # If normalization failed, try URL
        if extracted_date:
            return pd.Series([extracted_date, True])
        else:
            return pd.Series([original_date, False])

# Apply function to each row
df[["new_date", "date_modified"]] = df.apply(fix_date, axis=1)

# Save cleaned dataset
df.to_csv("gold-dataset-cleaned.csv", index=False)

print("✅ Cleaning complete. Saved as gold-dataset-cleaned.csv")
print(df[["Dates", "new_date", "date_modified"]].head())

C:\Users\AbdoZ\AppData\Local\Temp\ipykernel_13108\494257450.py:25: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(date_str, errors="coerce").strftime("%Y-%m-%d")


✅ Cleaning complete. Saved as gold-dataset-cleaned.csv
        Dates    new_date  date_modified
0  28-01-2016  2016-01-28          False
1  13-09-2017  2017-09-13          False
2  26-07-2016  2016-07-26          False
3  28-02-2018  2018-02-28          False
4  06-09-2017  2017-06-09          False
